# The Warehouse Floor: SARSA vs Q-Learning

**A hands-on introduction to Temporal Difference Reinforcement Learning**

---

## The Scenario

You are programming a warehouse robot to carry packages from the **Loading Dock** to the **Shipping Bay**. The floor has a dangerous conveyor belt along the bottom row and a chemical storage zone in the middle — plus scattered equipment that blocks certain paths.

The robot has no map. It learns purely from trial and error. You will teach it two strategies — **SARSA** and **Q-Learning** — and discover they learn *very different routes* through the same warehouse.

| Concept | Where |
|---|---|
| Epsilon-greedy exploration | Exercise 1 |
| **SARSA**  on-policy TD control | Exercise 2 |
| **Q-Learning**  off-policy TD control | Exercise 3 |
| Why the same warehouse yields different routes | Comparison section |
| Generalization across warehouse layouts | Multi-config test |
| Stochastic transitions (wet floors) | Bonus section |

---
*Estimated time: 60–90 minutes*

In [ ]:
#@title ## Part 1 — Setup
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
import json as _json
import random as _rnd
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams.update({'figure.facecolor':'#fafafa','axes.facecolor':'#ffffff','axes.grid':True,'grid.alpha':0.25,'font.size':11})

print('Imports ready.')

## Part 2 — The Warehouse Environment

The warehouse is a **5 × 12 grid** with two major hazard zones plus scattered equipment:

```
 Row 0:  ·  ·  ·  ☠  ·  ·  ·  ·  ☠  ·  ·  ·   ← Safe aisle (with obstacles)
 Row 1:  ·  ·  ·  ·  ·  ·  ·  ·  ·  ·  ·  ·   ← Upper corridor
 Row 2:  ·  ·  ·  ☠  ☠  ☠  ☠  ☠  ☠  ·  ·  ·   ← Chemical storage
 Row 3:  ·  ·  ·  ·  ·  ·  ·  ·  ·  ·  ·  ·   ← Narrow corridor (risky!)
 Row 4:  S  ☠  ☠  ☠  ☠  ☠  ☠  ☠  ☠  ☠  ☠  G   ← Conveyor belt
```

**Two routes from S to G:**

| Route | Via | Steps | Risk |
|---|---|---|---|
| Safe aisle | Rows 0–1 | ~19 | Low — but must weave around obstacles |
| Narrow corridor | Row 3 | ~13 | High — sandwiched between chemical storage and conveyor belt |


The environment above is implemented as the class `WarehouseEnv`. It constructs the grid, places the hazard cells (conveyor belt along row 4, chemical storage in row 2), and adds obstacle cells in row 0 depending on the chosen configuration (A, B, or C).

The agent can take one of **4 actions** at each step:

*   Up
*   Right
*   Down
*   Left

If an action would move the agent off the grid, it stays in place.

The class exposes 2 methods.

`reset()` sets the agent back to the start cell and returns that state. This is called at the beginning of each episode, or after the agent falls into a hazard.

`step(action)` applies the chosen action and returns 3 values: the **new state** (a tuple `(row, col)`), the **reward**, and a boolean **done** indicating whether the episode has ended (the agent reached the goal).

The reward structure is what drives learning: each normal step costs **−1** (encouraging shorter paths), stepping into a hazard gives **−100** (and resets the agent to Start without ending the episode), and reaching the goal G gives **0** (no penalty, episode ends). The agent's objective is to maximize cumulative reward, which in practice means reaching the goal in as few steps as possible while avoiding hazards.

Read through the environment code below.

In [ ]:
# ---------------------------------------------------------------------------
# Warehouse Environment  (provided, read through, no changes needed!)
# ---------------------------------------------------------------------------

ACTION_NAMES = ['Up', 'Right', 'Down', 'Left']
ACTION_ARROWS = ['\u2191', '\u2192', '\u2193', '\u2190']
ACTION_DELTAS = [(-1, 0), (0, 1), (1, 0), (0, -1)]
N_ACTIONS = 4


class WarehouseEnv:
    """5x12 warehouse: conveyor belt bottom, chemical storage middle,
    scattered obstacles in safe aisle. Config controls obstacle positions."""

    def __init__(self, config='A'):
        self.rows = 5
        self.cols = 12
        self.start = (4, 0)
        self.goal  = (4, 11)
        self.config = config

        self.cliffs = set()
        for c in range(1, 11):  self.cliffs.add((4, c))   # conveyor belt
        for c in range(3, 9):   self.cliffs.add((2, c))   # chemical storage

        obstacles = {
            'A': [(0, 3), (0, 8)],
            'B': [(0, 5), (0, 9)],
            'C': [(0, 2), (0, 6), (0, 10)],
        }
        for pos in obstacles.get(config, obstacles['A']):
            self.cliffs.add(tuple(pos))

        self.state = self.start

    def reset(self):
        self.state = self.start
        return self.state

    def step(self, action):
        dr, dc = ACTION_DELTAS[action]
        new_r = max(0, min(self.rows - 1, self.state[0] + dr))
        new_c = max(0, min(self.cols - 1, self.state[1] + dc))
        ns = (new_r, new_c)
        if ns in self.cliffs:
            self.state = self.start
            return self.start, -100, False
        elif ns == self.goal:
            self.state = ns
            return ns, 0, True
        else:
            self.state = ns
            return ns, -1, False


env = WarehouseEnv()
print(f'Grid   : {env.rows} x {env.cols}')
print(f'Start  : {env.start}')
print(f'Goal   : {env.goal}')
print(f'Hazards: {len(env.cliffs)} cells')
print('\nEnvironment ready.')

In [ ]:
# @title Visualization Helpers
# ---------------------------------------------------------------------------
# HTML grid rendering, policy display, path display, animation
# ---------------------------------------------------------------------------

_TILE = {
    'floor':'background:linear-gradient(145deg,#d5dbe0,#c8ced3)',
    'wet'  :'background:linear-gradient(145deg,#b3e5fc,#81d4fa)',
    'start':'background:linear-gradient(145deg,#3498db,#2980b9)',
    'goal' :'background:linear-gradient(145deg,#2ecc71,#27ae60)',
    'cliff':'background:repeating-linear-gradient(45deg,#e74c3c,#e74c3c 4px,#c0392b 4px,#c0392b 8px)',
}
_ARROWS_HTML = ['&#x2191;','&#x2192;','&#x2193;','&#x2190;']


def _cell_type(env, r, c):
    s = (r, c)
    if s == env.start: return 'start'
    if s == env.goal:  return 'goal'
    if s in env.cliffs: return 'cliff'
    if s in getattr(env, 'wet_zones', set()): return 'wet'
    return 'floor'


def _build_grid_html(env, title='', q_table=None, path=None, agent_pos=None):
    uid = ''.join(_rnd.choices('abcdefghijklmnopqrstuvwxyz', k=6))
    path_set = set(tuple(s) for s in path) if path else set()

    h = '<div style="font-family:\'Segoe UI\',system-ui,sans-serif;max-width:700px;margin:8px auto">'
    if title:
        h += f'<div style="padding:8px 14px;background:linear-gradient(135deg,#1a1a2e,#16213e);color:#eee;border-radius:8px 8px 0 0;font-size:13px;font-weight:700">{title}</div>'
    h += f'<div style="display:grid;grid-template-columns:repeat({env.cols},1fr);gap:2px;padding:6px;background:#0f3460;'
    h += 'border-radius:' + ('0' if title else '8px 8px') + ' 0 0">'

    for r in range(env.rows):
        for c in range(env.cols):
            ct = _cell_type(env, r, c)
            style = _TILE[ct] + ';aspect-ratio:1;display:flex;align-items:center;justify-content:center;border-radius:3px;font-size:12px'
            content = ''

            if ct == 'start':
                content = '<span style="color:#fff;font-weight:700;font-size:13px">S</span>'
            elif ct == 'goal':
                content = '<span style="color:#fff;font-weight:700;font-size:13px">G</span>'
            elif ct == 'cliff':
                content = '<span style="opacity:0.5;font-size:10px">&#x2620;</span>'
            elif ct == 'wet':
                content = '<span style="opacity:0.5;font-size:9px">&#x1F4A7;</span>'

            if q_table and (r,c) in q_table and ct not in ('cliff','goal'):
                best = int(np.argmax(q_table[(r,c)]))
                content = f'<span style="color:#2c3e50;font-size:16px;font-weight:700">{_ARROWS_HTML[best]}</span>'

            if path and (r,c) in path_set and ct not in ('cliff',):
                style += ';box-shadow:inset 0 0 0 3px rgba(52,152,219,0.5)'

            if agent_pos and (r,c) == agent_pos:
                content = '<span style="font-size:18px">&#x1F916;</span>'

            h += f'<div style="{style}">{content}</div>'

    h += '</div>'
    h += '<div style="display:flex;gap:12px;padding:6px 14px;background:#f4f6f8;border-radius:0 0 8px 8px;font-size:11px;color:#555;flex-wrap:wrap">'
    h += '<span>&#x1F7E6; Start</span><span>&#x1F7E9; Goal</span><span>&#x1F7E5; Hazard</span>'
    if any(_cell_type(env,r,c)=='wet' for r in range(env.rows) for c in range(env.cols)):
        h += '<span>&#x1F4A7; Wet</span>'
    h += '<span>&#x2B1C; Floor</span></div></div>'
    return h


def show_policy(q_table, env, title='Learned Policy'):
    return HTML(_build_grid_html(env, title=title, q_table=q_table))

def show_path(q_table, env, title='Greedy Path'):
    path, reward = run_greedy_episode(q_table, env)
    t = f'{title}  ({len(path)-1} steps, reward {reward})'
    return HTML(_build_grid_html(env, title=t, path=path, agent_pos=path[-1]))

def run_greedy_episode(q_table, env, max_steps=300):
    state = env.reset()
    path, total = [state], 0
    for _ in range(max_steps):
        if state in q_table:
            action = int(np.argmax(q_table[state]))
        else:
            action = np.random.randint(N_ACTIONS)
        state, reward, done = env.step(action)
        path.append(state)
        total += reward
        if done: break
    return path, total

def plot_curves(rewards_dict, window=50, title='Learning Curves'):
    fig, ax = plt.subplots(figsize=(10, 5))
    for name, rewards in rewards_dict.items():
        c = '#2980b9' if 'SARSA' in name else '#e67e22'
        sm = np.convolve(rewards, np.ones(window)/window, 'valid')
        ax.plot(sm, label=name, color=c, linewidth=2)
    ax.set_xlabel('Episode'); ax.set_ylabel(f'Reward ({window}-ep avg)')
    ax.set_title(title, fontsize=13, fontweight='bold'); ax.legend(fontsize=12)
    plt.tight_layout(); plt.show()



def animate_agent(q_table, env, title='Agent', interval=400):
    path, total_reward = run_greedy_episode(q_table, env)
    uid = ''.join(_rnd.choices('abcdefghijklmnopqrstuvwxyz', k=8))
    gi = [[_cell_type(env,r,c) for c in range(env.cols)] for r in range(env.rows)]
    has_wet = any(gi[r][c]=='wet' for r in range(env.rows) for c in range(env.cols))
    wet_legend = '<span>&#x1F4A7; Wet</span>' if has_wet else ''

    html = f"""
<div id="wh-{uid}" style="font-family:'Segoe UI',system-ui,sans-serif;max-width:700px;margin:12px auto;user-select:none">
 <div style="display:flex;justify-content:space-between;align-items:center;padding:10px 16px;background:linear-gradient(135deg,#1a1a2e,#16213e);color:#eee;border-radius:10px 10px 0 0;font-size:14px">
  <span style="font-weight:700">&#x1F916; {title}</span>
  <span>Step <b id="st-{uid}">0</b>/{len(path)-1}</span>
  <span>Reward: <b id="rw-{uid}">0</b></span>
 </div>
 <div style="position:relative;background:#0f3460;padding:6px;overflow:hidden">
  <div id="g-{uid}" style="display:grid;grid-template-columns:repeat({env.cols},1fr);gap:2px"></div>
  <div id="b-{uid}" style="position:absolute;display:flex;align-items:center;justify-content:center;font-size:22px;z-index:20;pointer-events:none;transition:left .28s cubic-bezier(.4,0,.2,1),top .28s cubic-bezier(.4,0,.2,1);filter:drop-shadow(0 2px 5px rgba(0,0,0,.5))">&#x1F916;</div>
  <svg id="sv-{uid}" style="position:absolute;top:6px;left:6px;width:calc(100% - 12px);height:calc(100% - 12px);z-index:10;pointer-events:none"></svg>
 </div>
 <div style="display:flex;align-items:center;gap:6px;padding:8px 12px;background:#e8edf2;flex-wrap:wrap">
  <button onclick="W{uid}.rst()" style="all:unset;cursor:pointer;padding:4px 8px;border-radius:6px;background:#ddd;font-size:15px">&#x23EE;</button>
  <button onclick="W{uid}.prv()" style="all:unset;cursor:pointer;padding:4px 8px;border-radius:6px;background:#ddd;font-size:15px">&#x25C0;</button>
  <button id="pl-{uid}" onclick="W{uid}.tog()" style="all:unset;cursor:pointer;padding:4px 10px;border-radius:6px;background:#3498db;color:#fff;font-size:15px">&#x25B6;</button>
  <button onclick="W{uid}.nxt()" style="all:unset;cursor:pointer;padding:4px 8px;border-radius:6px;background:#ddd;font-size:15px">&#x25B6;</button>
  <button onclick="W{uid}.end()" style="all:unset;cursor:pointer;padding:4px 8px;border-radius:6px;background:#ddd;font-size:15px">&#x23ED;</button>
  <span style="margin-left:10px;font-size:11px;color:#555">Speed</span>
  <input type="range" min="80" max="800" value="{interval}" style="width:80px;accent-color:#3498db" oninput="W{uid}.spd(this.value)">
 </div>
 <div style="display:flex;gap:12px;padding:6px 12px;background:#f4f6f8;border-radius:0 0 10px 10px;font-size:11px;color:#555;flex-wrap:wrap">
  <span>&#x1F7E6; Start</span><span>&#x1F7E9; Goal</span><span>&#x1F7E5; Hazard</span>{wet_legend}<span>&#x2B1C; Floor</span><span>&#x1F535; Trail</span>
 </div>
</div>
<script>
(function(){{var u='{uid}',P={_json.dumps(path)},R={total_reward},G={_json.dumps(gi)},NR={env.rows},NC={env.cols};
var st={{'floor':'background:linear-gradient(145deg,#d5dbe0,#c8ced3)','wet':'background:linear-gradient(145deg,#b3e5fc,#81d4fa)','start':'background:linear-gradient(145deg,#3498db,#2980b9)','goal':'background:linear-gradient(145deg,#2ecc71,#27ae60)','cliff':'background:repeating-linear-gradient(45deg,#e74c3c,#e74c3c 4px,#c0392b 4px,#c0392b 8px)'}};
var lb={{'start':'S','goal':'G'}},ic={{'cliff':'\\u2620','wet':'\\uD83D\\uDCA7'}};
var gE=document.getElementById('g-'+u),bE=document.getElementById('b-'+u),svE=document.getElementById('sv-'+u);
for(var r=0;r<NR;r++)for(var c=0;c<NC;c++){{var d=document.createElement('div'),t=G[r][c];
d.style.cssText=st[t]+';aspect-ratio:1;display:flex;align-items:center;justify-content:center;border-radius:3px;font-size:11px;font-weight:700';
if(lb[t]){{d.textContent=lb[t];d.style.color='#fff';d.style.fontSize='13px';d.style.textShadow='0 1px 2px rgba(0,0,0,.4)';}}
else if(ic[t]){{d.innerHTML=ic[t];d.style.opacity='.5';d.style.fontSize='10px';}}
gE.appendChild(d);}}
var fr=0,pl=false,tm=null,iv={interval};
function pos(r,c){{var w=gE.offsetWidth/NC,h=gE.offsetHeight/NR;bE.style.left=(6+c*w+1)+'px';bE.style.top=(6+r*h+1)+'px';bE.style.width=(w-2)+'px';bE.style.height=(h-2)+'px';}}
function trail(n){{svE.innerHTML='';if(n<1)return;var w=gE.offsetWidth/NC,h=gE.offsetHeight/NR;
for(var i=0;i<n;i++){{var a=P[i],b=P[i+1],l=document.createElementNS('http://www.w3.org/2000/svg','line');
l.setAttribute('x1',a[1]*w+w/2);l.setAttribute('y1',a[0]*h+h/2);l.setAttribute('x2',b[1]*w+w/2);l.setAttribute('y2',b[0]*h+h/2);
var o=.15+.6*(i/Math.max(1,n-1));l.setAttribute('stroke','rgba(52,152,219,'+o+')');l.setAttribute('stroke-width','3');l.setAttribute('stroke-linecap','round');svE.appendChild(l);}}
for(var i=0;i<n;i++){{var p=P[i],d=document.createElementNS('http://www.w3.org/2000/svg','circle');
d.setAttribute('cx',p[1]*w+w/2);d.setAttribute('cy',p[0]*h+h/2);d.setAttribute('r','3');
d.setAttribute('fill','rgba(52,152,219,'+(0.2+0.6*i/Math.max(1,n))+')');svE.appendChild(d);}}}}
function sf(f){{fr=Math.max(0,Math.min(P.length-1,f));pos(P[fr][0],P[fr][1]);trail(fr);
document.getElementById('st-'+u).textContent=fr;document.getElementById('rw-'+u).textContent=fr===P.length-1?R:-fr;
if(fr===P.length-1){{bE.style.transform='scale(1.15)';setTimeout(function(){{bE.style.transform='scale(1)'}},300);}}}}
function tk(){{if(fr<P.length-1)sf(fr+1);else stp();}}
function go(){{if(fr>=P.length-1)sf(0);pl=true;document.getElementById('pl-'+u).innerHTML='\\u23F8';tm=setInterval(tk,iv);}}
function stp(){{pl=false;document.getElementById('pl-'+u).innerHTML='\\u25B6';clearInterval(tm);}}
window['W'+u]={{tog:function(){{pl?stp():go()}},rst:function(){{stp();sf(0)}},end:function(){{stp();sf(P.length-1)}},prv:function(){{stp();sf(fr-1)}},nxt:function(){{stp();sf(fr+1)}},spd:function(v){{iv=+v;if(pl){{clearInterval(tm);tm=setInterval(tk,iv);}}}}}};
setTimeout(function(){{sf(0)}},80);
window.addEventListener('resize',function(){{pos(P[fr][0],P[fr][1]);trail(fr);}});
}})();
</script>"""
    return HTML(html)


# Quick test
display(HTML(_build_grid_html(WarehouseEnv(), title='Warehouse — Config A')))
print('Visualisation helpers ready.')


## How the Agent Learns

In the exercises that follow, we compare 3 approaches to action selection in this warehouse.

First, in Part 3, the agent picks actions **uniformly at random**. It has no knowledge of the environment and simply chooses one of the 4 directions with equal probability at every step. This serves as a baseline to see how poorly an uninformed agent performs.

Then, in Exercises 2 and 3, we implement 2 TD learning algorithms: **SARSA** and **Q-Learning**. In both cases the agent runs for a fixed number of episodes. Within each episode, it takes steps through the warehouse, selects actions using the **epsilon-greedy** strategy (Exercise 1), observes the reward, and updates its Q-table accordingly. The difference lies in how that update is computed. SARSA uses the action actually selected for the next step, Q-Learning uses the best possible action.

Over thousands of episodes the Q-values converge and the agent learns a policy. We will see that despite using the same environment and the same **epsilon-greedy** strategy, the two algorithms arrive at fundamentally different routes.

## Part 3 — Explore the Environment

Before any learning, we let the agent act purely at random: at each step it picks one of the 4 actions (Up, Right, Down, Left) with equal probability. Watch how often it stumbles into hazards.

In [ ]:
env = WarehouseEnv()
for ep in range(5):
    state = env.reset(); total, steps, hits = 0, 0, 0
    for _ in range(300):
        action = np.random.randint(N_ACTIONS)
        state, reward, done = env.step(action)
        total += reward; steps += 1
        if reward == -100: hits += 1
        if done: break
    status = 'Delivered' if done else 'Timed out'
    print(f'Episode {ep+1}: {status}  |  Steps: {steps:>4}  |  Hazard falls: {hits}  |  Reward: {total}')
print('\nRandom agent crashes constantly. Time to learn!')

## Part 4 Key Concepts: Temporal Difference Learning

Both SARSA and Q-Learning are **Temporal Difference (TD)** methods.

### Q-Values

$Q(s, a)$ estimates how good it is to take action $a$ in state $s$.

### The TD Update

$$Q(s,a) \leftarrow Q(s,a) + \alpha \Big[\underbrace{r + \gamma V_{\text{next}}}_{\text{TD target}} - Q(s,a)\Big]$$

### The One Difference

| Algorithm | $V_{\text{next}}$ | Meaning |
|---|---|---|
| **SARSA** | $Q(s', a')$  the action actually taken | On-policy: learns what we *do* |
| **Q-Learning** | $\max_{a'} Q(s', a')$  the best possible | Off-policy: learns what is *optimal* |

### Epsilon-Greedy

- With probability $\varepsilon$: random action
- With probability $1 - \varepsilon$: best known action

---
## 🎯 Exercise 1: Epsilon-Greedy Action Selection

Implement epsilon-greedy. Both algorithms use this.

> **Hint:** `np.random.random()` gives a float in $[0, 1)$. `np.argmax(array)` gives the index of the largest element.

In [ ]:
def epsilon_greedy(q_table: dict, state: tuple, epsilon: float) -> int:
    """Select an action using the epsilon-greedy policy.

    With probability epsilon, pick a random action (explore).
    With probability 1 - epsilon, pick the action with the highest
    Q-value for the current state (exploit).

    Args:
        q_table: dictionary mapping state -> array of Q-values (one per action)
        state:   current state as a tuple (row, col)
        epsilon: probability of choosing a random action, float in [0, 1]

    Returns:
        action: integer in {0, 1, 2, 3} corresponding to Up, Right, Down, Left
    """
    if state not in q_table:
        q_table[state] = np.zeros(N_ACTIONS)

    # 🎯 YOUR CODE HERE
    # Step 1: Generate a random number between 0 and 1
    # Step 2: If it is less than epsilon, return a random action (exploration)
    # Step 3: Otherwise, return the action with the highest Q-value (exploitation)
    # --- 🎯🎯🎯🎯 ---


In [ ]:
# Verification
_q = {(0,0): np.array([1.0, 5.0, 2.0, 3.0])}
assert all(epsilon_greedy(_q, (0,0), 0.0) == 1 for _ in range(200))
counts = [0]*4
for _ in range(4000): counts[epsilon_greedy(_q, (0,0), 1.0)] += 1
assert all(c > 500 for c in counts)
print('All checks passed!')

---
## 🎯 Exercise 2 — SARSA Update

$$Q(s, a) \leftarrow Q(s, a) + \alpha \Big[ r + \gamma \, Q(s', a') - Q(s, a) \Big]$$

$a'$ is the **actual next action** chosen by epsilon-greedy — this makes SARSA **on-policy**.

Map the formula to the code variables:

$$\texttt{td_target} = r + \gamma \, Q(s', a')$$

$$\texttt{td_error} = \texttt{td_target} - Q(s, a)$$

$$Q(s, a) \leftarrow Q(s, a) + \alpha \cdot \texttt{td_error}$$

In [ ]:
def sarsa_update(q_table, state, action, reward, next_state, next_action, alpha, gamma):
    if state not in q_table: 
        q_table[state] = np.zeros(N_ACTIONS)
    if next_state not in q_table: 
        q_table[next_state] = np.zeros(N_ACTIONS)

    # 🎯 YOUR CODE HERE
    td_target = ...
    td_error = ...
    q_table[state][action] += 0.0 # replace
    # --- 🎯🎯🎯🎯 ---


In [ ]:
#@title #### SARSA check (run this cell)
_q = {(0,0): np.array([0.,0.,0.,0.]), (0,1): np.array([1.,2.,3.,4.])}
sarsa_update(_q, (0,0), 1, -1.0, (0,1), 2, alpha=0.1, gamma=0.9)
assert abs(_q[(0,0)][1] - 0.17) < 1e-9
print(f'SARSA check passed: Q = {_q[(0,0)][1]:.4f}')

### SARSA Training Loop

In [ ]:
def train_sarsa(n_episodes=5000, alpha=0.1, gamma=1.0,
                eps_start=0.3, eps_end=0.01, verbose=True):
    """Train with SARSA using epsilon decay."""
    env = WarehouseEnv()
    q_table, rewards_log = {}, []
    for ep in range(n_episodes):
        epsilon = eps_start - (eps_start - eps_end) * ep / max(1, n_episodes - 1)
        state = env.reset()
        action = epsilon_greedy(q_table, state, epsilon)
        total = 0
        for _ in range(500):
            next_state, reward, done = env.step(action)
            next_action = epsilon_greedy(q_table, next_state, epsilon)
            sarsa_update(q_table, state, action, reward,
                         next_state, next_action, alpha, gamma)
            state, action = next_state, next_action
            total += reward
            if done: break
        rewards_log.append(total)
        if verbose and (ep+1) % max(1, n_episodes//5) == 0:
            avg = np.mean(rewards_log[max(0, len(rewards_log)-500):])
            print(f'  Ep {ep+1:>6}/{n_episodes}  |  eps={epsilon:.3f}  |  Avg reward: {avg:>8.1f}')
    return q_table, rewards_log

print('Training function ready.')

In [ ]:
print('Training SARSA ...')
sarsa_q, sarsa_rewards = train_sarsa(n_episodes=3000)
print(f'\nDone - {len(sarsa_q)} states learned.')

In [ ]:
display(show_policy(sarsa_q, WarehouseEnv(), 'SARSA - Learned Policy'))
display(show_path(sarsa_q, WarehouseEnv(), 'SARSA - Greedy Path'))

### Watch the SARSA Agent

In [ ]:
animate_agent(sarsa_q, WarehouseEnv(), title='SARSA Agent', interval=400)

---
## 🎯 Exercise 3 — Q-Learning Update

$$Q(s, a) \leftarrow Q(s, a) + \alpha \Big[ r + \gamma \, \max_{a'} Q(s', a') - Q(s, a) \Big]$$

Uses $\max_{a'} Q(s', a')$ instead of the actual next action — this makes Q-Learning **off-policy**.

Map the formula to the code variables:

$$\texttt{td\_target} = r + \gamma \, \max_{a'} Q(s', a')$$

$$\texttt{td\_error} = \texttt{td\_target} - Q(s, a)$$

$$Q(s, a) \leftarrow Q(s, a) + \alpha \cdot \texttt{td\_error}$$

In [ ]:
def q_learning_update(q_table, state, action, reward, next_state, alpha, gamma):
    if state not in q_table: q_table[state] = np.zeros(N_ACTIONS)
    if next_state not in q_table: q_table[next_state] = np.zeros(N_ACTIONS)

    # 🎯 YOUR CODE HERE
    td_target = ...
    td_error  = ...
    q_table[state][action] += 0.0 # replace
    # --- 🎯🎯🎯🎯 ---


In [ ]:
#@title ####Q-Learning check (run this cell)
_q = {(0,0): np.array([0.,0.,0.,0.]), (0,1): np.array([1.,2.,3.,4.])}
q_learning_update(_q, (0,0), 1, -1.0, (0,1), alpha=0.1, gamma=0.9)
assert abs(_q[(0,0)][1] - 0.26) < 1e-9
print(f'Q-Learning check passed: Q = {_q[(0,0)][1]:.4f}')

### Q-Learning Training Loop

In [ ]:
def train_q_learning(n_episodes=5000, alpha=0.1, gamma=1.0,
                     eps_start=0.3, eps_end=0.01, verbose=True):
    """Train with Q-Learning using epsilon decay."""
    env = WarehouseEnv()
    q_table, rewards_log = {}, []
    for ep in range(n_episodes):
        epsilon = eps_start - (eps_start - eps_end) * ep / max(1, n_episodes - 1)
        state = env.reset()
        total = 0
        for _ in range(500):
            action = epsilon_greedy(q_table, state, epsilon)
            next_state, reward, done = env.step(action)
            q_learning_update(q_table, state, action, reward,
                              next_state, alpha, gamma)
            state = next_state
            total += reward
            if done: break
        rewards_log.append(total)
        if verbose and (ep+1) % max(1, n_episodes//5) == 0:
            avg = np.mean(rewards_log[max(0, len(rewards_log)-500):])
            print(f'  Ep {ep+1:>6}/{n_episodes}  |  eps={epsilon:.3f}  |  Avg reward: {avg:>8.1f}')
    return q_table, rewards_log

print('Training function ready.')

In [ ]:
print('Training Q-Learning ...')
ql_q, ql_rewards = train_q_learning(n_episodes=3000)
print(f'\nDone - {len(ql_q)} states learned.')

In [ ]:
display(show_policy(ql_q, WarehouseEnv(), 'Q-Learning - Learned Policy'))
display(show_path(ql_q, WarehouseEnv(), 'Q-Learning - Greedy Path'))

### Watch the Q-Learning Agent

In [ ]:
animate_agent(ql_q, WarehouseEnv(), title='Q-Learning Agent', interval=400)

---
## Part 5 — The Big Comparison

We now train both algorithms from scratch under identical conditions: same number of episodes (3000), same learning rate ($\alpha = 0.1$), same discount factor ($\gamma = 1.0$), same constant exploration rate ($\varepsilon = 0.1$), and same warehouse layout (Config A). The only difference is the update rule: SARSA uses $Q(s', a')$, Q-Learning uses $\max_{a'} Q(s', a')$. This controlled setup isolates the effect of on-policy vs off-policy learning.

In [ ]:
N_EP, ALPHA, GAMMA, EPS = 3000, 0.1, 1.0, 0.1
print(f'Head-to-head  |  {N_EP} episodes  |  alpha={ALPHA}  gamma={GAMMA}  eps={EPS}')
print('=' * 55)
print('\n[SARSA]')
sarsa_q, sarsa_r = train_sarsa(N_EP, ALPHA, GAMMA, EPS)
print('\n[Q-Learning]')
ql_q, ql_r = train_q_learning(N_EP, ALPHA, GAMMA, EPS)
print('\nDone.')

In [ ]:
plot_curves({'SARSA': sarsa_r, 'Q-Learning': ql_r}, window=20, title='Training Reward')
print('SARSA policy:')
display(show_policy(sarsa_q, WarehouseEnv(), 'SARSA - Learned Policy'))
print('Q-Learning policy:')
display(show_policy(ql_q, WarehouseEnv(), 'Q-Learning - Learned Policy'))
print('SARSA path:')
display(show_path(sarsa_q, WarehouseEnv(), 'SARSA - Greedy Path'))
print('Q-Learning path:')
display(show_path(ql_q, WarehouseEnv(), 'Q-Learning - Greedy Path'))

In [ ]:
# @title Q-Value Heatmaps

# ---------------------------------------------------------------------------
# Q-Value Heatmaps
# ---------------------------------------------------------------------------

def plot_q_heatmaps(sarsa_q, ql_q, env, figsize=(16, 5)):
    """Side-by-side heatmaps of max Q-value per cell."""
    fig, axes = plt.subplots(1, 2, figsize=figsize)

    for ax, q_table, name, cmap in zip(axes,
                                        [sarsa_q, ql_q],
                                        ['SARSA', 'Q-Learning'],
                                        ['YlOrRd_r', 'YlOrRd_r']):
        grid = np.full((env.rows, env.cols), np.nan)
        for r in range(env.rows):
            for c in range(env.cols):
                if (r, c) in q_table:
                    grid[r, c] = np.max(q_table[(r, c)])

        mask = np.zeros_like(grid, dtype=bool)
        for (r, c) in env.cliffs:
            mask[r, c] = True

        masked = np.ma.array(grid, mask=mask)
        im = ax.imshow(masked, cmap=cmap, interpolation='nearest', aspect='equal')

        # Overlay cliff cells in red hatching
        for (r, c) in env.cliffs:
            ax.add_patch(patches.Rectangle((c - 0.5, r - 0.5), 1, 1,
                         linewidth=0, facecolor='#e74c3c', alpha=0.6))

        # Mark start and goal
        ax.plot(env.start[1], env.start[0], 's', color='#3498db',
                markersize=14, markeredgecolor='white', markeredgewidth=2)
        ax.plot(env.goal[1], env.goal[0], 's', color='#2ecc71',
                markersize=14, markeredgecolor='white', markeredgewidth=2)

        # Annotate values
        for r in range(env.rows):
            for c in range(env.cols):
                if not mask[r, c] and not np.isnan(grid[r, c]):
                    val = grid[r, c]
                    color = 'white' if val < (np.nanmin(grid) + (np.nanmax(grid) - np.nanmin(grid)) * 0.4) else '#2c3e50'
                    ax.text(c, r, f'{val:.0f}', ha='center', va='center',
                            fontsize=7, fontweight='bold', color=color)

        ax.set_title(f'{name} — max Q(s,a) per cell', fontsize=13, fontweight='bold')
        ax.set_xticks(range(env.cols))
        ax.set_yticks(range(env.rows))
        ax.set_xlabel('Column')
        ax.set_ylabel('Row')
        fig.colorbar(im, ax=ax, shrink=0.8, label='Q-value')

    plt.suptitle('Maximum Q-Value per State — SARSA vs Q-Learning',
             fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

    # Step comparison table
    sarsa_path, sarsa_reward = run_greedy_episode(sarsa_q, WarehouseEnv())
    ql_path, ql_reward = run_greedy_episode(ql_q, WarehouseEnv())
    print(f'\n{"Metric":<25} {"SARSA":>10} {"Q-Learning":>12}')
    print('-' * 50)
    print(f'{"Greedy path steps":<25} {len(sarsa_path)-1:>10} {len(ql_path)-1:>12}')
    print(f'{"Greedy path reward":<25} {sarsa_reward:>10.0f} {ql_reward:>12.0f}')
    print(f'{"States visited (training)":<25} {len(sarsa_q):>10} {len(ql_q):>12}')

    # Highlight the key insight
    sarsa_row3_vals = [np.max(sarsa_q[(3, c)]) for c in range(env.cols) if (3, c) in sarsa_q]
    ql_row3_vals = [np.max(ql_q[(3, c)]) for c in range(env.cols) if (3, c) in ql_q]
    print(f'\n{"Avg Q in Row 3 (corridor)":<25} {np.mean(sarsa_row3_vals):>10.1f} {np.mean(ql_row3_vals):>12.1f}')
    print(f'\n→ SARSA assigns significantly lower Q-values to states adjacent to hazards.')
    print('  Because SARSA is on-policy, its Q-values reflect the expected return under')
    print('  the epsilon-greedy policy including the probability of exploratory actions')
    print('  that lead into hazard cells. States near the conveyor belt or chemical storage')
    print('  carry a high expected penalty from these random moves, so the learned policy')
    print('  routes around them entirely.')
    print('')
    print('  Q-Learning, being off-policy, evaluates each state under the assumption of')
    print('  optimal future behavior. Since the greedy policy would never step into a hazard,')
    print('  the Q-values near dangerous zones remain relatively high, and the algorithm')
    print('  has no reason to avoid the shorter corridor route.')


plot_q_heatmaps(sarsa_q, ql_q, WarehouseEnv())

### What Should You See?

1. **SARSA takes the safe aisle** (rows 0-1), weaving around obstacles. ~19 steps.
2. **Q-Learning takes the narrow corridor** (row 3). ~13 steps but risky.
3. SARSA avoids hazards during training. Q-Learning crashes more but learns the optimal greedy policy.

> **Key insight:** On-policy (SARSA) = cautious. Off-policy (Q-Learning) = optimistic.

---
## Part 6 — Generalization: Testing on New Layouts

A good policy should transfer to similar warehouses. Let's test our policies (trained on Config A) on two unseen configurations where the obstacles in the safe aisle have moved.

In [ ]:
for cfg in ['A', 'B', 'C']:
    display(HTML(_build_grid_html(WarehouseEnv(cfg), title=f'Config {cfg}')))

In [ ]:
# @title ## Results for different configurations
print('Evaluating policies trained on Config A across all configs:\n')
print(f'{"":>12} {"Config A (train)":>18} {"Config B (new)":>18} {"Config C (new)":>18}')
print('-' * 70)
for name, q in [('SARSA', sarsa_q), ('Q-Learning', ql_q)]:
    results = []
    for cfg in ['A', 'B', 'C']:
        wins, total_r = 0, 0
        for _ in range(500):
            env_t = WarehouseEnv(cfg); s = env_t.reset(); ep_r = 0
            for __ in range(200):
                a = int(np.argmax(q[s])) if s in q else np.random.randint(N_ACTIONS)
                s, r, d = env_t.step(a); ep_r += r
                if d: break
            if s == env_t.goal: wins += 1
            total_r += ep_r
        results.append(f'{wins/500:>5.0%} ({total_r/500:>6.1f})')
    print(f'{name:>12} {results[0]:>18} {results[1]:>18} {results[2]:>18}')
print('\nQ-Learning\'s corridor route ignores row 0 obstacles entirely,')
print('so it transfers perfectly. SARSA\'s safe-aisle route is tailored')
print('to Config A and crashes on new obstacle positions.')

In [ ]:
# @title ## Part 7 — Hyperparameter Experiments
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for alpha in [0.05, 0.1, 0.2, 0.5]:
    _, sr = train_sarsa(n_episodes=3000, alpha=alpha, verbose=False)
    _, qr = train_q_learning(n_episodes=3000, alpha=alpha, verbose=False)
    w = 50
    axes[0].plot(np.convolve(sr, np.ones(w)/w, 'valid'), label=f'alpha={alpha}', lw=1.5)
    axes[1].plot(np.convolve(qr, np.ones(w)/w, 'valid'), label=f'alpha={alpha}', lw=1.5)
for ax, nm in zip(axes, ['SARSA', 'Q-Learning']):
    ax.set_title(f'{nm} - Learning Rate', fontweight='bold')
    ax.set_xlabel('Episode'); ax.set_ylabel('Reward'); ax.legend(); ax.set_ylim([-200, 0])
plt.tight_layout(); plt.show()

In [ ]:
#@title ##
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for eps in [0.01, 0.1, 0.2, 0.4]:
    _, sr = train_sarsa(n_episodes=3000, eps_start=eps, eps_end=eps, verbose=False)
    _, qr = train_q_learning(n_episodes=3000, eps_start=eps, eps_end=eps, verbose=False)
    w = 50
    axes[0].plot(np.convolve(sr, np.ones(w)/w, 'valid'), label=f'eps={eps}', lw=1.5)
    axes[1].plot(np.convolve(qr, np.ones(w)/w, 'valid'), label=f'eps={eps}', lw=1.5)
for ax, nm in zip(axes, ['SARSA', 'Q-Learning']):
    ax.set_title(f'{nm} - Exploration Rate', fontweight='bold')
    ax.set_xlabel('Episode'); ax.set_ylabel('Reward'); ax.legend(); ax.set_ylim([-400, 0])
plt.tight_layout(); plt.show()

---
## Part 8 — Bonus: Slippery Warehouse (Frozen Lake-style)

Now we match the **OpenAI Gym Frozen Lake** design exactly:

- **Sparse random hazards** (4 holes in a 5×8 grid, ~10% density — like Frozen Lake 8×8)
- **Globally slippery floor**: every tile is slippery (1/3 intended, 1/3 each perpendicular)
- **Terminal hazards**: step on one and the episode ends (no reset-to-start loops)
- **Frozen Lake rewards**: 0 everywhere, +1 only at goal

Each configuration has randomly placed hazards. We train independently on each and show the agent learns to navigate around the holes despite the slippery floor.


In [ ]:
class SlipperyWarehouseEnv:
    """Frozen Lake-style: sparse random hazards, globally slippery, terminal.
    Uses 5×8 grid (proven to converge in 20k episodes with 1/3 slip)."""

    def __init__(self, seed=42, n_hazards=4, rows=5, cols=8):
        rng = np.random.RandomState(seed)
        self.rows, self.cols = rows, cols
        self.start = (0, 0)
        self.goal = (rows - 1, cols - 1)

        free = [(r, c) for r in range(rows) for c in range(cols)
                if (r, c) != self.start and (r, c) != self.goal]
        idxs = rng.choice(len(free), size=n_hazards, replace=False)
        self.cliffs = {free[i] for i in idxs}

        self.wet_zones = {(r, c) for r in range(rows) for c in range(cols)
                          if (r, c) not in self.cliffs and (r, c) != self.goal}
        self.state = self.start

    def reset(self):
        self.state = self.start
        return self.state

    def step(self, action):
        action = np.random.choice([(action-1)%4, action, (action+1)%4])
        dr, dc = ACTION_DELTAS[action]
        new_r = max(0, min(self.rows - 1, self.state[0] + dr))
        new_c = max(0, min(self.cols - 1, self.state[1] + dc))
        ns = (new_r, new_c)

        if ns in self.cliffs:
            self.state = ns
            return ns, 0.0, True
        elif ns == self.goal:
            self.state = ns
            return ns, 1.0, True
        else:
            self.state = ns
            return ns, 0.0, False


In [ ]:
env_test = SlipperyWarehouseEnv(seed=42)
print(f'Grid: {env_test.rows}x{env_test.cols}, Hazards: {len(env_test.cliffs)}, Wet: {len(env_test.wet_zones)}')

# Check slippery mechanics
env_test.state = (0, 1)
landings = set()
for _ in range(200):
    env_test.state = (0, 1)
    ns, _, _ = env_test.step(1)  # try RIGHT
    landings.add(ns)
assert len(landings) > 1, f'Slippery should produce varied outcomes, got {landings}'
print(f'Passed! Slippery verified: RIGHT from (0,1) landed at {sorted(landings)}')


In [ ]:
def train_slippery(algo, env, n_ep=20000, alpha=0.1, eps_start=1.0, eps_end=0.01):
    """Train on a slippery environment with epsilon decay."""
    q, rews = {}, []
    for ep in range(n_ep):
        epsilon = eps_start - (eps_start - eps_end) * ep / max(1, n_ep - 1)
        s = env.reset(); total = 0
        if algo == 'sarsa': a = epsilon_greedy(q, s, epsilon)
        for _ in range(200):
            if algo == 'sarsa':
                ns, r, d = env.step(a)
                na = epsilon_greedy(q, ns, epsilon)
                sarsa_update(q, s, a, r, ns, na, alpha, 0.99)
                s, a = ns, na
            else:
                a = epsilon_greedy(q, s, epsilon)
                ns, r, d = env.step(a)
                q_learning_update(q, s, a, r, ns, alpha, 0.99)
                s = ns
            total += r
            if d: break
        rews.append(total)
    return q, rews

In [ ]:
#@title ###Run this cell multiple times. Each run generates a NEW random warehouse!
seed = np.random.randint(0, 10000)
env = SlipperyWarehouseEnv(seed=seed, n_hazards=4)

print(f'Random warehouse (seed={seed})')
display(HTML(_build_grid_html(env, title=f'Slippery Warehouse (seed={seed})')))

print(f'\nTraining SARSA (20k episodes)...')
sarsa_q_slip, sarsa_r_slip = train_slippery('sarsa', env, n_ep=20000)

print(f'Training Q-Learning (20k episodes)...')
ql_q_slip, ql_r_slip = train_slippery('qlearning', env, n_ep=20000)

# Evaluate both
for name, q in [('SARSA', sarsa_q_slip), ('Q-Learning', ql_q_slip)]:
    wins = 0
    for _ in range(100):
        e = SlipperyWarehouseEnv(seed=seed, n_hazards=4)
        s = e.reset()
        for __ in range(200):
            a = int(np.argmax(q[s])) if s in q else np.random.randint(N_ACTIONS)
            s, r, d = e.step(a)
            if d: break
        if r > 0: wins += 1
    print(f'  {name} greedy success: {wins}/100')

plot_curves({'SARSA': sarsa_r_slip, 'Q-Learning': ql_r_slip},
            window=200, title=f'Slippery Warehouse (seed={seed})')

print('\nSARSA:')
display(animate_agent(sarsa_q_slip, SlipperyWarehouseEnv(seed=seed), f'SARSA — seed={seed}'))
print('\nQ-Learning:')
display(animate_agent(ql_q_slip, SlipperyWarehouseEnv(seed=seed), f'Q-Learning — seed={seed}'))

---
## Summary

| | SARSA | Q-Learning |
|---|---|---|
| **Update uses** | $Q(s', a')$ — action actually taken | $\max_{a'} Q(s', a')$ — best possible action |
| **Type** | On-policy | Off-policy |
| **Learned route** | Safe aisle (~19 steps) | Narrow corridor (~13 steps) |
| **Training reward** | Higher (fewer crashes) | Lower (more crashes during exploration) |
| **Greedy deployment** | Suboptimal (conservative) | Optimal (shortest path) |


---

### What We Learned

Both SARSA and Q-Learning are Temporal Difference (TD) methods that learn Q-values estimates of the expected cumulative reward for taking a given action in a given state. They share the same TD update structure but differ in a single term.

**SARSA** updates using the action actually taken next:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \Big[ r + \gamma \, Q(s', a') - Q(s, a) \Big]$$

**Q-Learning** updates using the best possible next action:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \Big[ r + \gamma \, \max_{a'} Q(s', a') - Q(s, a) \Big]$$

### When to Use Which

The choice between SARSA and Q-Learning is ultimately about the **cost of mistakes during training**. If the agent is learning in a simulation where crashes are free, Q-Learning is preferable because it converges to the optimal policy. If the agent is learning on real hardware: a warehouse robot, a quadruped like ANYmal, a chemical process controller then each training mistake has a real cost, and SARSA's conservative learning may be the safer choice.

It is also worth noting that SARSA's policy is tailored to the specific exploration rate used during training. If $\varepsilon$ is large, SARSA learns a very cautious policy; if $\varepsilon$ is small, it approaches Q-Learning's behavior. Q-Learning's result is independent of $\varepsilon$ (given sufficient exploration for convergence), which is why it generalized better across warehouse configurations in Part 6.

---

### The Management Analogy

**SARSA**: *"Take the longer route. You'll make mistakes while learning, so stay safe."*

**Q-Learning**: *"The corridor is fastest. You'll break things while training, but once trained, it's optimal."*

Neither is universally better. The right choice depends on the **cost of mistakes during learning** and **how much the environment might change**.

---

### Connection to What Comes Next

The REINFORCE algorithm, which you will see in the next session, takes a fundamentally different approach. Instead of learning Q-values and deriving a policy from them, it parameterizes the policy directly and updates it using gradient ascent on the expected return. This avoids the need for a Q-table entirely, which becomes essential in environments with continuous action spaces where "take the max over all actions" is no longer possible.

---

*Congratulations!*